# Air Quality Patterns in African Cities
### Data Analytics Capstone — Group 5

**Client:** United Nations Environment Programme (UNEP)
**Team:** Samuel Tokoye, Paul Kibet Miningwa, Winnie Odoyo, Gloria Simiyu Wandabwa
**Data source:** OpenAQ v3 API
**Cities studied:** Nairobi, Kampala, Kigali, Addis Ababa, Johannesburg, Lagos
**Study period:** 2021–present

## Project brief

Air pollution is an increasing public-health concern across African cities. UNEP has commissioned
this analysis to evaluate air-quality patterns across six selected cities, in order to guide
future intervention priorities.

## Research questions

1. Which city recorded the highest average PM2.5 concentration during the study period?
2. Which pollutants are monitored most frequently across the selected cities?
3. Which cities experience the greatest seasonal variation in PM2.5 concentrations?
4. How frequently do PM2.5 measurements exceed the WHO recommended guideline values?
5. Which monitoring stations have the most complete and reliable datasets?
6. Which cities should be prioritised for air-quality intervention programmes, and why?

Every recommendation in this notebook is directly supported by evidence obtained during the
analysis below.


## 1. Data Acquisition: API Documentation

**API:** OpenAQ v3 (`https://api.openaq.org/v3`)
**Authentication:** API key required, sent via the `X-API-Key` header

### Endpoints used

**`GET /v3/locations`** — identify monitoring stations within each city.

| Parameter | Value used | Notes |
|---|---|---|
| `coordinates` | `{lat},{lon}` per city center | e.g. Nairobi: `-1.286389,36.817223` |
| `radius` | `25000` (metres) | 25km — the maximum radius v3 allows |
| `limit` | `100` | Sufficient for all cities studied |

Each location response includes an embedded `sensors` array (pollutant type, sensor ID) and
`datetimeFirst`/`datetimeLast` reporting window, so a separate `/sensors` call was unnecessary.

**`GET /v3/sensors/{sensor_id}/days`** — pull daily-averaged PM2.5 measurements for each active sensor.

| Parameter | Value used | Notes |
|---|---|---|
| `date_from` | `2021-01-01` | Start of study window |
| `date_to` | current date | End of study window |
| `limit` | `1000` | Max page size |
| `page` | incremented until a page returns <1000 results | `meta.found` returned inconsistent formats across responses, so page-size was used as the reliable pagination stop condition instead |

### Preprocessing rules applied during acquisition

- **Pollutant scope:** full time-series pulled for **PM2.5 only**. Other pollutants were catalogued
  by station frequency (for Q2) but not pulled as time-series — none of the six client questions
  require trend data for pollutants other than PM2.5.
- **"Active sensor" definition:** a sensor counts as active if its station's `datetimeLast` falls
  within 2 years of the pull date. Only active sensors had measurement history pulled.
- **Known limitation — Johannesburg:** 7 PM2.5 sensors found, only 1 active. Johannesburg's overall
  monitoring is also skewed toward SO2/PM10/CO rather than PM2.5, likely reflecting
  industrial/regulatory monitoring infrastructure rather than the low-cost sensor networks
  driving PM2.5 density in Kampala, Lagos, and Nairobi. Retained in the study with this limitation
  documented rather than dropped.

### Files produced by acquisition (see `scripts/`)

| File | Contents |
|---|---|
| `data/raw/{city}_locations.json` | Raw station metadata per city |
| `data/raw/pm25_sensors_summary.json` | Extracted PM2.5 sensors, active/inactive flag |
| `data/raw/{city}_measurements.json` | Daily-averaged PM2.5 readings per active sensor |
| `data/raw/pollutant_frequency.csv` | Station counts per pollutant type per city |


## 2. Data Loading

Load the outputs of the acquisition scripts (see `scripts/`) so the rest of this notebook works from saved data, not live API calls.

In [3]:
import json
import os
import pandas as pd

CITIES = ["nairobi", "kampala", "kigali", "addis_ababa", "johannesburg", "lagos"]
DATA_DIR = "../data/raw"


### 2.1 Station metadata per city

In [4]:
locations = {}
for city in CITIES:
    with open(f"{DATA_DIR}/{city}_locations.json") as f:
        locations[city] = json.load(f)
    print(f"{city}: {locations[city]['meta']['found']} stations found")


nairobi: 16 stations found
kampala: 36 stations found
kigali: 4 stations found
addis_ababa: 9 stations found
johannesburg: 12 stations found
lagos: 61 stations found


### 2.2 PM2.5 sensor summary (active/inactive flag)

In [5]:
with open(f"{DATA_DIR}/pm25_sensors_summary.json") as f:
    sensors_df = pd.DataFrame(json.load(f))

print(f"Total PM2.5 sensors: {len(sensors_df)}")
print(f"Active sensors: {sensors_df['is_active'].sum()}")
sensors_df.groupby("city")["is_active"].agg(["sum", "count"]).rename(
    columns={"sum": "active", "count": "total"}
)


Total PM2.5 sensors: 133
Active sensors: 119


,active,total
city,,
addis_ababa,7,9
johannesburg,1,7
kampala,35,36
kigali,3,4
lagos,61,61
nairobi,12,16


### 2.3 Daily PM2.5 measurements per city

In [6]:
measurements = {}
for city in CITIES:
    path = f"{DATA_DIR}/{city}_measurements.json"
    if os.path.exists(path):
        with open(path) as f:
            measurements[city] = json.load(f)
        total_days = sum(len(s["daily_averages"]) for s in measurements[city])
        print(f"{city}: {len(measurements[city])} active sensor(s), {total_days} total daily records")
    else:
        print(f"{city}: no measurements file found")


nairobi: 12 active sensor(s), 2108 total daily records
kampala: 35 active sensor(s), 7271 total daily records
kigali: 3 active sensor(s), 1418 total daily records
addis_ababa: 7 active sensor(s), 2851 total daily records
johannesburg: 1 active sensor(s), 3 total daily records
lagos: 61 active sensor(s), 9064 total daily records


### 2.4 Pollutant monitoring frequency (Q2 data source)

In [7]:
pollutant_freq = pd.read_csv(f"{DATA_DIR}/pollutant_frequency.csv")
pollutant_freq[pollutant_freq["station_count"] > 0].sort_values(
    ["city", "station_count"], ascending=[True, False]
)


,city,pollutant,station_count
77,addis_ababa,PM2.5,9
100,johannesburg,SO₂,10
97,johannesburg,PM10,9
98,johannesburg,PM2.5,7
90,johannesburg,CO,5
94,johannesburg,O₃,5
91,johannesburg,NO,4
92,johannesburg,NOx,4
93,johannesburg,NO₂,4
95,johannesburg,PM0.3 count,1


## 3. Data Quality Assessment


This section audits data quality before cleaning, using the raw PM2.5 daily files loaded in Section 2.

### QA checks performed

1. Missing values: missing `value` records and missing coverage fields by city and year.
2. Duplicate records: repeated `(city, sensor_id, date_utc)` daily rows.
3. Outliers:
   - Physically implausible values (`value < 0` or `value > 500` µg/m³).
   - Statistical outliers using a city-level IQR rule.
4. Completeness: station-level continuity (`actual_days / expected_days`) and average OpenAQ `percentComplete`.
5. Consistency: units, averaging period labels/intervals, timezone offsets, and date ordering checks.

All QA outputs are shown as tables below and will directly inform Section 4 cleaning decisions.

In [8]:
import numpy as np
from IPython.display import display

rows = []
for city, city_stations in measurements.items():
    for station in city_stations:
        for day in station.get("daily_averages", []):
            period = day.get("period", {})
            dt_from = period.get("datetimeFrom", {})
            dt_to = period.get("datetimeTo", {})
            parameter = day.get("parameter", {})
            coverage = day.get("coverage", {})

            rows.append({
                "city": city,
                "station_id": station.get("station_id"),
                "station_name": station.get("station_name"),
                "sensor_id": station.get("sensor_id"),
                "date_utc": dt_from.get("utc"),
                "date_local": dt_from.get("local"),
                "date_to_utc": dt_to.get("utc"),
                "value": day.get("value"),
                "unit": parameter.get("units"),
                "parameter_name": parameter.get("name"),
                "period_label": period.get("label"),
                "period_interval": period.get("interval"),
                "expected_count": coverage.get("expectedCount"),
                "observed_count": coverage.get("observedCount"),
                "percent_complete": coverage.get("percentComplete"),
                "has_flags": day.get("flagInfo", {}).get("hasFlags"),
            })

qa_df = pd.DataFrame(rows)
qa_df["date_utc"] = pd.to_datetime(qa_df["date_utc"], errors="coerce", utc=True)
qa_df["date_to_utc"] = pd.to_datetime(qa_df["date_to_utc"], errors="coerce", utc=True)
qa_df["year"] = qa_df["date_utc"].dt.year

print(f"Total PM2.5 daily rows in scope: {len(qa_df):,}")
print(f"Cities covered: {qa_df['city'].nunique()} | Stations covered: {qa_df['station_id'].nunique()} | Sensors covered: {qa_df['sensor_id'].nunique()}")




Total PM2.5 daily rows in scope: 22,715
Cities covered: 6 | Stations covered: 119 | Sensors covered: 119


In [10]:
# Missing Values:
missing_city = (
    qa_df.groupby("city", as_index=False)
    .agg(
        total_rows=("value", "size"),
        missing_value_rows=("value", lambda s: s.isna().sum()),
        missing_percent_complete=("percent_complete", lambda s: s.isna().sum()),
        missing_date_rows=("date_utc", lambda s: s.isna().sum()),
    )
)
missing_city["missing_value_pct"] = (100 * missing_city["missing_value_rows"] / missing_city["total_rows"]).round(2)
missing_city["missing_percent_complete_pct"] = (
    100 * missing_city["missing_percent_complete"] / missing_city["total_rows"]
).round(2)
display(missing_city.sort_values("missing_value_pct", ascending=False))

missing_year = (
    qa_df.groupby(["city", "year"], as_index=False)
    .agg(
        total_rows=("value", "size"),
        missing_value_rows=("value", lambda s: s.isna().sum()),
    )
)
missing_year["missing_value_pct"] = (100 * missing_year["missing_value_rows"] / missing_year["total_rows"]).round(2)
display(missing_year.sort_values(["city", "year"]))

,city,total_rows,missing_value_rows,missing_percent_complete,missing_date_rows,missing_value_pct,missing_percent_complete_pct
0,addis_ababa,2851,366,0,0,12.84,0.0
2,kampala,7271,250,0,0,3.44,0.0
4,lagos,9064,17,0,0,0.19,0.0
1,johannesburg,3,0,0,0,0.00,0.0
3,kigali,1418,0,0,0,0.00,0.0
5,nairobi,2108,0,0,0,0.00,0.0


,city,year,total_rows,missing_value_rows,missing_value_pct
0,addis_ababa,2020,1,0,0.00
1,addis_ababa,2021,468,23,4.91
2,addis_ababa,2022,691,231,33.43
3,addis_ababa,2023,587,78,13.29
4,addis_ababa,2024,659,31,4.70
5,addis_ababa,2025,326,3,0.92
6,addis_ababa,2026,119,0,0.00
7,johannesburg,2026,3,0,0.00
8,kampala,2020,1,1,100.00
9,kampala,2021,356,36,10.11


In [11]:
# Identify duplicate records based on city, station_id, sensor_id, and date_utc.
dup_mask = qa_df.duplicated(subset=["city", "sensor_id", "date_utc"], keep=False)
dup_rows = qa_df[dup_mask].sort_values(["city", "sensor_id", "date_utc"] )
duplicate_summary = (
    qa_df.groupby("city", as_index=False)
    .apply(lambda g: pd.Series({
        "duplicate_rows": g.duplicated(subset=["sensor_id", "date_utc"]).sum(),
        "total_rows": len(g),
    }))
    .reset_index(drop=True)
)
duplicate_summary["duplicate_pct"] = (100 * duplicate_summary["duplicate_rows"] / duplicate_summary["total_rows"]).round(4)
display(duplicate_summary.sort_values("duplicate_rows", ascending=False))

if dup_rows.empty:
    print("No duplicate (city, sensor_id, date_utc) rows detected.")
else:
    print("Sample duplicate rows:")
    display(dup_rows.head(20))

,city,duplicate_rows,total_rows,duplicate_pct
0,addis_ababa,0,2851,0.0
1,johannesburg,0,3,0.0
2,kampala,0,7271,0.0
3,kigali,0,1418,0.0
4,lagos,0,9064,0.0
5,nairobi,0,2108,0.0


No duplicate (city, sensor_id, date_utc) rows detected.


In [12]:
# Outliers check using IQR method
qa_df["implausible_physical"] = (qa_df["value"] < 0) | (qa_df["value"] > 500)

# City-level IQR rule (computed on non-missing values).
city_q = qa_df.groupby("city")["value"].quantile([0.25, 0.75]).unstack()
city_q.columns = ["q1", "q3"]
city_q["iqr"] = city_q["q3"] - city_q["q1"]
city_q["lower_iqr"] = city_q["q1"] - 1.5 * city_q["iqr"]
city_q["upper_iqr"] = city_q["q3"] + 1.5 * city_q["iqr"]

qa_df = qa_df.merge(city_q[["lower_iqr", "upper_iqr"]], left_on="city", right_index=True, how="left")
qa_df["iqr_outlier"] = (qa_df["value"] < qa_df["lower_iqr"]) | (qa_df["value"] > qa_df["upper_iqr"] )
qa_df.loc[qa_df["value"].isna(), "iqr_outlier"] = False

outlier_summary = (
    qa_df.groupby("city", as_index=False)
    .agg(
        total_rows=("value", "size"),
        physical_outliers=("implausible_physical", "sum"),
        iqr_outliers=("iqr_outlier", "sum"),
    )
)
outlier_summary["physical_outlier_pct"] = (100 * outlier_summary["physical_outliers"] / outlier_summary["total_rows"]).round(4)
outlier_summary["iqr_outlier_pct"] = (100 * outlier_summary["iqr_outliers"] / outlier_summary["total_rows"]).round(2)
display(outlier_summary.sort_values("iqr_outlier_pct", ascending=False))


,city,total_rows,physical_outliers,iqr_outliers,physical_outlier_pct,iqr_outlier_pct
4,lagos,9064,72,584,0.7944,6.44
2,kampala,7271,33,392,0.4539,5.39
5,nairobi,2108,0,87,0.0000,4.13
0,addis_ababa,2851,0,93,0.0000,3.26
3,kigali,1418,0,33,0.0000,2.33
1,johannesburg,3,0,0,0.0000,0.00


In [13]:
# Completeness check: median coverage ratio and mean daily percent complete per city.
station_completeness = (
    qa_df.groupby(["city", "station_id", "station_name", "sensor_id"], as_index=False)
    .agg(
        first_date=("date_utc", "min"),
        last_date=("date_utc", "max"),
        actual_days=("date_utc", "nunique"),
        mean_percent_complete=("percent_complete", "mean"),
        non_missing_values=("value", lambda s: s.notna().sum()),
    )
)
station_completeness["expected_days"] = (
    (station_completeness["last_date"] - station_completeness["first_date"]).dt.days + 1
)
station_completeness["coverage_ratio"] = (
    station_completeness["actual_days"] / station_completeness["expected_days"]
).replace([np.inf, -np.inf], np.nan)

# Practical quality bands for this project.
station_completeness["quality_band"] = pd.cut(
    station_completeness["coverage_ratio"],
    bins=[-0.01, 0.5, 0.8, 0.95, 1.0],
    labels=["very low", "low", "medium", "high"],
    include_lowest=True,
 )

city_completeness = (
    station_completeness.groupby("city", as_index=False)
    .agg(
        stations=("station_id", "count"),
        median_coverage_ratio=("coverage_ratio", "median"),
        mean_daily_percent_complete=("mean_percent_complete", "mean"),
    )
)
city_completeness["median_coverage_ratio"] = city_completeness["median_coverage_ratio"].round(3)
city_completeness["mean_daily_percent_complete"] = city_completeness["mean_daily_percent_complete"].round(2)
display(city_completeness.sort_values("median_coverage_ratio", ascending=False))

print("Lowest-coverage stations (top 15):")
display(
    station_completeness.sort_values(["coverage_ratio", "mean_percent_complete"], ascending=[True, True])
    [["city", "station_name", "sensor_id", "actual_days", "expected_days", "coverage_ratio", "mean_percent_complete", "quality_band"]]
    .head(15)
)

,city,stations,median_coverage_ratio,mean_daily_percent_complete
0,addis_ababa,7,1.000,34.25
1,johannesburg,1,1.000,87.67
3,kigali,3,1.000,73.49
5,nairobi,12,0.984,81.71
2,kampala,35,0.937,48.71
4,lagos,61,0.878,53.19


Lowest-coverage stations (top 15):


,city,station_name,sensor_id,actual_days,expected_days,coverage_ratio,mean_percent_complete,quality_band
81,lagos,"Miri Air - Diamond School, Ogijo",15915968,6,152,0.039474,44.833333,very low
28,kampala,airqo_g5449,14507651,7,103,0.067961,7.142857,very low
57,lagos,Oshodi,13520831,2,17,0.117647,4.000000,very low
104,lagos,LAMATA Place,17066590,2,11,0.181818,31.000000,very low
66,lagos,Ikeja Bus Terminal,13928964,50,223,0.224215,7.320000,very low
71,lagos,Ojuelegba QBC,14010806,7,30,0.233333,31.571429,very low
51,lagos,Shagisha - Magodo,13418561,114,320,0.356250,60.964912,very low
118,nairobi,Providence Academy,15427115,46,125,0.368000,76.304348,very low
107,nairobi,Nairobi RR,2002532,371,973,0.381295,93.035040,very low
61,lagos,LAMATA Place,13557362,37,88,0.420455,18.405405,very low


In [14]:
# Consistency checks: check for any negative values in the 'value' column and any missing units or parameter names.

units_check = qa_df.groupby("city", as_index=False)["unit"].nunique().rename(columns={"unit": "unique_units"})
period_label_check = qa_df.groupby("city", as_index=False)["period_label"].nunique().rename(columns={"period_label": "unique_period_labels"})
period_interval_check = qa_df.groupby("city", as_index=False)["period_interval"].nunique().rename(columns={"period_interval": "unique_period_intervals"})

consistency_summary = units_check.merge(period_label_check, on="city").merge(period_interval_check, on="city")
display(consistency_summary)

print("Unit values observed:")
display(
    qa_df[["city", "unit"]].drop_duplicates().sort_values(["city", "unit"]).reset_index(drop=True)
)

print("Period labels and intervals observed:")
display(
    qa_df[["city", "period_label", "period_interval"]]
    .drop_duplicates()
    .sort_values(["city", "period_label", "period_interval"] )
    .reset_index(drop=True)
)

date_order_issues = qa_df[qa_df["date_to_utc"] < qa_df["date_utc"]]
print(f"Rows with date_to_utc earlier than date_utc: {len(date_order_issues)}")

,city,unique_units,unique_period_labels,unique_period_intervals
0,addis_ababa,1,1,1
1,johannesburg,1,1,1
2,kampala,1,1,1
3,kigali,1,1,1
4,lagos,1,1,1
5,nairobi,1,1,1


Unit values observed:


,city,unit
0,addis_ababa,µg/m³
1,johannesburg,µg/m³
2,kampala,µg/m³
3,kigali,µg/m³
4,lagos,µg/m³
5,nairobi,µg/m³


Period labels and intervals observed:


,city,period_label,period_interval
0,addis_ababa,1day,24:00:00
1,johannesburg,1day,24:00:00
2,kampala,1day,24:00:00
3,kigali,1day,24:00:00
4,lagos,1day,24:00:00
5,nairobi,1day,24:00:00


Rows with date_to_utc earlier than date_utc: 0


In [15]:
# Conclusion: The QA checks reveal that there are missing values, duplicate records, outliers, and completeness issues in the dataset.

qa_conclusion = missing_city[["city", "missing_value_pct"]].merge(
    duplicate_summary[["city", "duplicate_rows"]], on="city", how="left"
).merge(
    outlier_summary[["city", "physical_outliers", "iqr_outlier_pct"]], on="city", how="left"
).merge(
    city_completeness[["city", "median_coverage_ratio", "mean_daily_percent_complete"]], on="city", how="left"
)
display(qa_conclusion.sort_values(["missing_value_pct", "median_coverage_ratio"], ascending=[False, True]))


,city,missing_value_pct,duplicate_rows,physical_outliers,iqr_outlier_pct,median_coverage_ratio,mean_daily_percent_complete
0,addis_ababa,12.84,0,0,3.26,1.000,34.25
2,kampala,3.44,0,33,5.39,0.937,48.71
4,lagos,0.19,0,72,6.44,0.878,53.19
5,nairobi,0.00,0,0,4.13,0.984,81.71
1,johannesburg,0.00,0,0,0.00,1.000,87.67
3,kigali,0.00,0,0,2.33,1.000,73.49


## 4. Cleaning & Preprocessing


This section applies cleaning decisions directly informed by the QA findings in Section 3. Each 
decision is documented below before being applied in code.

**1. Missing values.**
- QA's outlier check found 105 physically-impossible readings (value <0 or >500 µg/m³): Lagos 72, 
  Kampala 33. These are reclassified as missing (NaN) here; not dropped as outliers; so they can't 
  wrongly anchor interpolation in the next step.
- Genuine gaps (QA found Addis Ababa 12.84%, Kampala 3.44%, Lagos 0.19% missing) are interpolated if 
  ≤3 consecutive days per sensor.
- Longer gaps are dropped, not fabricated. Concentrated in Addis Ababa 2022 and Kampala 2023 (33% 
  missing each); a bad year, not random noise.

In [16]:
# 1a. Reclassify QA's physically-implausible outliers as missing (before interpolation)
clean_df = qa_df.copy()
decisions_log = []

n_implausible = clean_df["implausible_physical"].sum()
clean_df.loc[clean_df["implausible_physical"], "value"] = np.nan
decisions_log.append(f"Missing (1a): reclassified {n_implausible} QA-flagged outliers as NaN (Lagos 72, Kampala 33).")
print(decisions_log[-1])

Missing (1a): reclassified 105 QA-flagged outliers as NaN (Lagos 72, Kampala 33).


In [17]:
# 1b. Interpolate gaps ≤3 days per sensor — using transform (safe across pandas versions)
clean_df = clean_df.sort_values(["city", "sensor_id", "date_utc"])
before_missing = clean_df["value"].isna()

clean_df["value"] = clean_df.groupby(["city", "sensor_id"])["value"].transform(
    lambda s: s.interpolate(method="linear", limit=3, limit_area="inside")
)

clean_df["value_interpolated_flag"] = before_missing & clean_df["value"].notna()

decisions_log.append(f"Missing (1b): interpolated {clean_df['value_interpolated_flag'].sum()} short-gap rows.")
print(decisions_log[-1])

Missing (1b): interpolated 246 short-gap rows.


In [18]:
# 1c. Drop rows with unfillable gaps
before = len(clean_df)
clean_df = clean_df.dropna(subset=["value"])
decisions_log.append(f"Missing (1c): dropped {before - len(clean_df)} rows with gaps >3 days.")
print(decisions_log[-1])

Missing (1c): dropped 492 rows with gaps >3 days.


In [19]:
# Run this immediately, in a NEW cell, right now:
print(clean_df.columns.tolist())
print(clean_df.shape)

['city', 'station_id', 'station_name', 'sensor_id', 'date_utc', 'date_local', 'date_to_utc', 'value', 'unit', 'parameter_name', 'period_label', 'period_interval', 'expected_count', 'observed_count', 'percent_complete', 'has_flags', 'year', 'implausible_physical', 'lower_iqr', 'upper_iqr', 'iqr_outlier', 'value_interpolated_flag']
(22223, 22)


**2. Duplicate records.** QA's `.duplicated()` check detected 0 duplicate `(city, sensor_id, 
date_utc)` rows. `.drop_duplicates()` is applied here as the actual removal step, 0 rows are expected 
to be removed, consistent with QA's finding; and is kept as a defensive safeguard in case a future 
re-pull of the API introduces duplicates that today's data doesn't have.

In [20]:
# 2. Duplicate records 
before = len(clean_df)
clean_df = clean_df.drop_duplicates(subset=["city", "sensor_id", "date_utc"])
removed = before - len(clean_df)
decisions_log.append(
    f"Duplicates: removed {removed} rows via drop_duplicates()\n"
    f"QA's .duplicated() check detected 0 duplicates; this matches that finding; "
    f"the step is retained as a defensive safeguard for future re-pulls."
)
print(decisions_log[-1])

Duplicates: removed 0 rows via drop_duplicates()
QA's .duplicated() check detected 0 duplicates; this matches that finding; the step is retained as a defensive safeguard for future re-pulls.


**3. Outliers.** With physically-implausible readings already removed under missing values (step 1a), 
this step covers only statistical (IQR) outliers among genuine readings. IQR bounds are recomputed 
here on the cleaned data, rather than reusing QA's original bounds, because QA's bounds were computed 
before the implausible values were removed and would otherwise be skewed by them. These IQR outliers 
(2–6% of rows per city, recomputed) are **flagged, not dropped**,  PM2.5 spikes can reflect real 
events (fires, dust, traffic) that matter directly to Q4's WHO exceedance analysis, and removing them 
would understate real pollution exposure.

In [21]:
# 3. Outliers: recompute IQR bounds on cleaned data, flag (don't drop) 
# Physically implausible readings were already removed under missing values (step 1a),
# so this step covers only statistical outliers among genuine remaining readings.
city_q = clean_df.groupby("city")["value"].quantile([0.25, 0.75]).unstack()
city_q.columns = ["q1", "q3"]
city_q["iqr"] = city_q["q3"] - city_q["q1"]
city_q["lower_iqr"] = city_q["q1"] - 1.5 * city_q["iqr"]
city_q["upper_iqr"] = city_q["q3"] + 1.5 * city_q["iqr"]

clean_df = clean_df.drop(columns=["lower_iqr", "upper_iqr", "iqr_outlier"], errors="ignore")
clean_df = clean_df.merge(city_q[["lower_iqr", "upper_iqr"]], left_on="city", right_index=True, how="left")
clean_df["iqr_outlier"] = (clean_df["value"] < clean_df["lower_iqr"]) | (clean_df["value"] > clean_df["upper_iqr"])

decisions_log.append(
    f"Outliers: recomputed IQR bounds on cleaned data ({clean_df['iqr_outlier'].sum()} rows"
    f"flagged, 2-6% per city); recalculated here rather than reusing QA's original bounds, since "
    f"those were computed before the implausible values were removed and would otherwise be skewed.\n "
    f"Flagged rows were RETAINED, not dropped: PM2.5 spikes can be real events relevant to Q4's "
    f"exceedance analysis."
)
print(decisions_log[-1])

Outliers: recomputed IQR bounds on cleaned data (1161 rowsflagged, 2-6% per city); recalculated here rather than reusing QA's original bounds, since those were computed before the implausible values were removed and would otherwise be skewed.
 Flagged rows were RETAINED, not dropped: PM2.5 spikes can be real events relevant to Q4's exceedance analysis.


**4. Completeness.** Each row carries forward its station's `coverage_ratio` and `quality_band` 
(very low / low / medium / high), taken directly from QA's Section 3 completeness check on the 
original raw data; this metric describes data *availability*, so it is not recalculated after 
cleaning. This lets later analysis phases decide whether to include or exclude low-reliability 
stations per question, rather than that decision being made silently here.

Within completeness, **Johannesburg is a documented limitation, not a drop**: QA found only 1 active 
PM2.5 sensor with 3 total daily records in the entire study window, reflecting OpenAQ's limited PM2.5 
coverage there (its monitoring infrastructure skews toward SO2/PM10/CO). Johannesburg is retained for 
Q1 (a single-point average) but flagged as unusable for any time-series question (Q3 seasonal 
variation, Q4 exceedance-over-time), since 3 data points cannot support that kind of analysis.

In [22]:
# 4. Completeness: carry forward station-level reliability info 
clean_df = clean_df.merge(
    station_completeness[["city", "sensor_id", "coverage_ratio", "quality_band"]],
    on=["city", "sensor_id"], how="left"
)
decisions_log.append(
    "Completeness: merged coverage_ratio and quality_band, taken directly from QA's Section 3 "
    "completeness check on the raw data (this metric describes original availability, so it is not "
    "recalculated post-cleaning), so downstream analysis can filter by station reliability per question."
)
decisions_log.append(
    "Completeness: Johannesburg RETAINED for Q1 only; flagged as UNRELIABLE for any "
    "time-series question (Q3, Q4), since QA found only 1 active sensor with 3 total daily records "
    "in the study window. This reflects OpenAQ's limited PM2.5 coverage there, not a cleaning artifact."
)
for line in decisions_log[-2:]:
    print(line)

Completeness: merged coverage_ratio and quality_band, taken directly from QA's Section 3 completeness check on the raw data (this metric describes original availability, so it is not recalculated post-cleaning), so downstream analysis can filter by station reliability per question.
Completeness: Johannesburg RETAINED for Q1 only; flagged as UNRELIABLE for any time-series question (Q3, Q4), since QA found only 1 active sensor with 3 total daily records in the study window. This reflects OpenAQ's limited PM2.5 coverage there, not a cleaning artifact.


**5. Consistency.** QA confirmed all six cities report a single unit (µg/m³) and a single period 
label/interval (`1day` / `24:00:00`); no standardization was required. This step re-confirms that 
finding still holds after cleaning, rather than repeating unnecessary transformatio

In [23]:
# --- 5. Consistency: re-confirm units and period labels after cleaning ---
units_ok = clean_df["unit"].nunique() == 1
period_ok = clean_df["period_label"].nunique() == 1 and clean_df["period_interval"].nunique() == 1

decisions_log.append(
    f"Consistency: re-confirmed after cleaning — single unit "
    f"('{clean_df['unit'].unique()[0]}': {units_ok}) and single period label/interval "
    f"('{clean_df['period_label'].unique()[0]}' / '{clean_df['period_interval'].unique()[0]}': {period_ok}). "
    f"No standardization needed; matches QA's original finding."
)
assert units_ok and period_ok, "Consistency check failed; investigate before saving master file."
print(decisions_log[-1])

Consistency: re-confirmed after cleaning — single unit ('µg/m³': True) and single period label/interval ('1day' / '24:00:00': True). No standardization needed; matches QA's original finding.


**Output:** `data/cleaned/air_quality_master.csv`; one row per city/station/sensor/day, with `pm25`, 
`coverage_ratio`, `quality_band`, `value_interpolated_flag`, and `iqr_outlier` columns available for 
downstream filtering.

In [24]:
# 6. Build final master DataFrame and save 
master_cols = [
    "city", "station_id", "station_name", "sensor_id", "date_utc", "date_local",
    "value", "unit", "coverage_ratio", "quality_band","year","has_flags",
    "value_interpolated_flag", "iqr_outlier", 
]
air_quality_master = clean_df[master_cols].rename(columns={"value": "pm25", "unit": "pm25_unit"})

import os
os.makedirs("../data/cleaned", exist_ok=True)
air_quality_master.to_csv("../data/cleaned/air_quality_master.csv", index=False)

print(f"Final master dataset: {len(air_quality_master):,} rows across "
      f"{air_quality_master['city'].nunique()} cities, "
      f"{air_quality_master['sensor_id'].nunique()} sensors")
air_quality_master.head()

Final master dataset: 22,223 rows across 6 cities, 119 sensors


,city,station_id,station_name,sensor_id,date_utc,date_local,pm25,pm25_unit,coverage_ratio,quality_band,year,has_flags,value_interpolated_flag,iqr_outlier
0,addis_ababa,9714,Addis Ababa Central,30190,2020-12-31 21:00:00+00:00,2021-01-01T00:00:00+03:00,14.5,µg/m³,0.884087,medium,2020,False,False,False
1,addis_ababa,9714,Addis Ababa Central,30190,2021-01-01 21:00:00+00:00,2021-01-02T00:00:00+03:00,9.5,µg/m³,0.884087,medium,2021,False,False,False
2,addis_ababa,9714,Addis Ababa Central,30190,2021-01-02 21:00:00+00:00,2021-01-03T00:00:00+03:00,21.3,µg/m³,0.884087,medium,2021,False,False,False
3,addis_ababa,9714,Addis Ababa Central,30190,2021-01-03 21:00:00+00:00,2021-01-04T00:00:00+03:00,14.4,µg/m³,0.884087,medium,2021,False,False,False
4,addis_ababa,9714,Addis Ababa Central,30190,2021-01-04 21:00:00+00:00,2021-01-05T00:00:00+03:00,15.1,µg/m³,0.884087,medium,2021,False,False,False


In [26]:
# Full preprocessing decisions log 
print("PREPROCESSING DECISIONS LOG:")
for i, line in enumerate(decisions_log, 1):
    print(f"{i}. {line}\n")

PREPROCESSING DECISIONS LOG:
1. Missing (1a): reclassified 105 QA-flagged outliers as NaN (Lagos 72, Kampala 33).

2. Missing (1b): interpolated 246 short-gap rows.

3. Missing (1c): dropped 492 rows with gaps >3 days.

4. Duplicates: removed 0 rows via drop_duplicates()
QA's .duplicated() check detected 0 duplicates; this matches that finding; the step is retained as a defensive safeguard for future re-pulls.

5. Outliers: recomputed IQR bounds on cleaned data (1161 rowsflagged, 2-6% per city); recalculated here rather than reusing QA's original bounds, since those were computed before the implausible values were removed and would otherwise be skewed.
 Flagged rows were RETAINED, not dropped: PM2.5 spikes can be real events relevant to Q4's exceedance analysis.

6. Completeness: merged coverage_ratio and quality_band, taken directly from QA's Section 3 completeness check on the raw data (this metric describes original availability, so it is not recalculated post-cleaning), so downstre

## 5. Exploratory Data Analysis & Statistical Techniques

**TODO:** Apply at least 3 statistical techniques answering Q1, Q3, Q4, Q5: descriptive/ranking (highest avg PM2.5), trend/seasonal analysis, WHO exceedance-rate analysis, completeness scoring.

In [24]:
import pandas as pd
from pathlib import Path
from scipy import stats

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Reuse Section 4 output in memory when available; otherwise load from saved cleaned file.
if "air_quality_master" in globals():
    df = air_quality_master.copy()
else:
    master_path = Path("../data/cleaned/air_quality_master.csv")
    df = pd.read_csv(master_path, parse_dates=["date_utc"] )

df["date_utc"] = pd.to_datetime(df["date_utc"], utc=True, errors="coerce")
df["month"] = df["date_utc"].dt.month
df["season"] = (df["month"] % 12 // 3) + 1

print(f"Rows: {len(df):,} | Cities: {df['city'].nunique()} | Stations: {df['station_id'].nunique()}")
print(f"Date range: {df['date_utc'].min().date()} to {df['date_utc'].max().date()}")
df.head()

Rows: 22,223 | Cities: 6 | Stations: 119
Date range: 2020-12-31 to 2026-08-06


,city,station_id,station_name,sensor_id,date_utc,date_local,pm25,pm25_unit,coverage_ratio,quality_band,year,has_flags,value_interpolated_flag,iqr_outlier,month,season
0,addis_ababa,9714,Addis Ababa Central,30190,2020-12-31 21:00:00+00:00,2021-01-01T00:00:00+03:00,14.50,µg/m³,0.88,medium,2020,False,False,False,12,1
1,addis_ababa,9714,Addis Ababa Central,30190,2021-01-01 21:00:00+00:00,2021-01-02T00:00:00+03:00,9.50,µg/m³,0.88,medium,2021,False,False,False,1,1
2,addis_ababa,9714,Addis Ababa Central,30190,2021-01-02 21:00:00+00:00,2021-01-03T00:00:00+03:00,21.30,µg/m³,0.88,medium,2021,False,False,False,1,1
3,addis_ababa,9714,Addis Ababa Central,30190,2021-01-03 21:00:00+00:00,2021-01-04T00:00:00+03:00,14.40,µg/m³,0.88,medium,2021,False,False,False,1,1
4,addis_ababa,9714,Addis Ababa Central,30190,2021-01-04 21:00:00+00:00,2021-01-05T00:00:00+03:00,15.10,µg/m³,0.88,medium,2021,False,False,False,1,1


## Data scope note
Q1, Q3, Q4, and Q5 use the cleaned PM2.5 dataset (`air_quality_master`) created in Section 4.

Q2 (pollutants monitored most frequently) is answered from `pollutant_freq`, which comes from the API-derived file
`data/raw/pollutant_frequency.csv` loaded in Section 2.4. This keeps pollutant-frequency analysis tied to the original
OpenAQ location metadata rather than the PM2.5-only cleaned table.

Johannesburg has very limited PM2.5 history (1 active sensor, 3 daily records), so it is retained but interpreted
with caution for time-series and reliability-heavy conclusions.

## Question 1. Which city recorded the highest average PM2.5 concentration?
**Technique: descriptive statistics (mean, median, std) + ranking**

In [25]:
# Question 1
q1 = df.groupby('city')['pm25'].agg(
    n_readings='count', mean_pm25='mean', median_pm25='median', std_pm25='std'
).sort_values('mean_pm25', ascending=False)
q1['rank'] = range(1, len(q1) + 1)
q1

,n_readings,mean_pm25,median_pm25,std_pm25,rank
city,,,,,
kampala,7113,52.58,43.20,43.80,1
johannesburg,3,44.70,39.70,9.90,2
kigali,1418,38.93,35.75,17.58,3
lagos,9042,38.62,28.20,51.99,4
addis_ababa,2539,26.10,23.80,15.80,5
nairobi,2108,23.15,17.90,22.50,6


In [26]:
top_city = q1.index[0]
print(f"Answer: {top_city.replace('_', ' ').title()} recorded the highest average PM2.5 "
      f"({q1.loc[top_city, 'mean_pm25']:.1f} µg/m³, n={q1.loc[top_city, 'n_readings']:,.0f} readings).")
print("Caveat: Johannesburg's mean is based on only 3 readings from 1 station and is not statistically reliable.")

Answer: Kampala recorded the highest average PM2.5 (52.6 µg/m³, n=7,113 readings).
Caveat: Johannesburg's mean is based on only 3 readings from 1 station and is not statistically reliable.


## 2. Which pollutants are monitored most frequently across the selected cities?
**Technique: descriptive/ranking statistics (station & sensor counts, record volume)**

In [27]:
# Q2 uses API-derived pollutant frequency data from Section 2.4
if "pollutant_freq" not in globals():
    pollutant_freq = pd.read_csv("../data/raw/pollutant_frequency.csv")

pollutant_col = "pollutant" if "pollutant" in pollutant_freq.columns else "parameter"

q2_city_pollutant = (
    pollutant_freq[pollutant_freq["station_count"] > 0]
    .sort_values(["city", "station_count"], ascending=[True, False])
    .copy()
)

q2_total_pollutants = (
    pollutant_freq.groupby(pollutant_col, as_index=False)["station_count"]
    .sum()
    .sort_values("station_count", ascending=False)
)

# Keep a city-level monitoring intensity table for later synthesis (Q6).
q2 = df.groupby("city").agg(
    n_stations=("station_id", "nunique"),
    n_sensors=("sensor_id", "nunique"),
    n_records=("pm25", "count")
).sort_values("n_records", ascending=False)

print("Most monitored pollutant types overall (across all selected cities):")
display(q2_total_pollutants.head(10))

print("Per-city pollutant monitoring ranking:")
display(q2_city_pollutant)

print("PM2.5 monitoring intensity by city (used later in synthesis):")
display(q2)

Most monitored pollutant types overall (across all selected cities):


,pollutant,station_count
14,PM2.5,133
17,Temperature (C),61
15,RH,60
13,PM10,58
12,PM1,51
11,PM0.3 count,47
0,Atmospheric pressure,10
16,SO₂,10
6,CO,6
9,NO₂,6


Per-city pollutant monitoring ranking:


,city,pollutant,station_count
77,addis_ababa,PM2.5,9
100,johannesburg,SO₂,10
97,johannesburg,PM10,9
98,johannesburg,PM2.5,7
90,johannesburg,CO,5
94,johannesburg,O₃,5
91,johannesburg,NO,4
92,johannesburg,NOx,4
93,johannesburg,NO₂,4
95,johannesburg,PM0.3 count,1


PM2.5 monitoring intensity by city (used later in synthesis):


,n_stations,n_sensors,n_records
city,,,
lagos,61,61,9042
kampala,35,35,7113
addis_ababa,7,7,2539
nairobi,12,12,2108
kigali,3,3,1418
johannesburg,1,1,3


In [28]:
top_pollutant = q2_total_pollutants.iloc[0]
pollutant_name_col = "pollutant" if "pollutant" in q2_total_pollutants.columns else "parameter"
print(
    f"Answer: {top_pollutant[pollutant_name_col]} is the most frequently monitored pollutant "
    f"({int(top_pollutant['station_count'])} station-monitor entries across the six cities)."
)

city_top = (
    q2_city_pollutant.groupby("city", as_index=False).first()
    .sort_values("station_count", ascending=False)
)
print("Top monitored pollutant by city:")
display(city_top[["city", pollutant_name_col, "station_count"]])

Answer: PM2.5 is the most frequently monitored pollutant (133 station-monitor entries across the six cities).
Top monitored pollutant by city:


,city,pollutant,station_count
4,lagos,PM2.5,61
2,kampala,PM2.5,36
5,nairobi,PM2.5,16
1,johannesburg,SO₂,10
0,addis_ababa,PM2.5,9
3,kigali,PM2.5,4


## 3. Which cities experience the greatest seasonal variation in PM2.5?
**Technique: time-series / seasonal trend analysis (monthly means + one-way ANOVA across months)**

In [29]:
monthly = df.groupby(['city', 'month'])['pm25'].mean().unstack('month')

seasonal_range = (monthly.max(axis=1) - monthly.min(axis=1)).sort_values(ascending=False)
seasonal_cv = (df.groupby('city')['pm25'].std() / df.groupby('city')['pm25'].mean()).sort_values(ascending=False)

q3 = pd.DataFrame({'monthly_mean_range': seasonal_range, 'coefficient_of_variation': seasonal_cv})
q3

,monthly_mean_range,coefficient_of_variation
city,,
addis_ababa,18.86,0.61
johannesburg,0.00,0.22
kampala,27.68,0.83
kigali,30.82,0.45
lagos,41.37,1.35
nairobi,29.55,0.97


In [30]:
# One-way ANOVA: is monthly PM2.5 significantly different within each city? (tests seasonality is real, not noise)
anova_rows = []
for city, g in df.groupby('city'):
    groups = [g.loc[g['month'] == m, 'pm25'].dropna() for m in sorted(g['month'].unique())]
    groups = [gr for gr in groups if len(gr) >= 2]
    if len(groups) >= 2:
        f_stat, p_val = stats.f_oneway(*groups)
        anova_rows.append({'city': city, 'f_stat': f_stat, 'p_value': p_val, 'significant_seasonality': p_val < 0.05})
anova_df = pd.DataFrame(anova_rows).set_index('city').sort_values('f_stat', ascending=False)
anova_df

,f_stat,p_value,significant_seasonality
city,,,
kigali,75.69,0.00,True
lagos,46.28,0.00,True
addis_ababa,37.61,0.00,True
kampala,27.27,0.00,True
nairobi,20.45,0.00,True


In [31]:
top_seasonal = seasonal_range.index[0]
print(f"Answer: {top_seasonal.replace('_',' ').title()} shows the greatest seasonal swing "
      f"(monthly mean range = {seasonal_range.iloc[0]:.1f} µg/m³).")
sig_cities = anova_df[anova_df['significant_seasonality']].index.tolist()
print(f"ANOVA confirms statistically significant month-to-month variation (p<0.05) in: "
      f"{', '.join(c.replace('_',' ').title() for c in sig_cities) if sig_cities else 'none of the cities'}.")

Answer: Lagos shows the greatest seasonal swing (monthly mean range = 41.4 µg/m³).
ANOVA confirms statistically significant month-to-month variation (p<0.05) in: Kigali, Lagos, Addis Ababa, Kampala, Nairobi.


## 4. How frequently do PM2.5 measurements exceed the WHO guideline?
**Technique: threshold / exceedance-rate analysis**

WHO Global Air Quality Guidelines (2021): 24-hour mean PM2.5 guideline = **15 µg/m³**;
annual mean guideline = **5 µg/m³**. Each row in the cleaned data is a daily station reading, so the
24-hour guideline is the relevant threshold for exceedance-rate calculation; the annual guideline is
used to test the yearly average.

In [32]:
WHO_24H = 15.0
WHO_ANNUAL = 5.0

q4 = df.groupby('city').agg(
    n_readings=('pm25', 'count'),
    exceed_24h=('pm25', lambda s: (s > WHO_24H).sum())
)
q4['exceedance_rate_24h_%'] = (q4['exceed_24h'] / q4['n_readings'] * 100)

annual_means = df.groupby('city')['pm25'].mean()
q4['annual_mean_pm25'] = annual_means
q4['exceeds_annual_guideline'] = annual_means > WHO_ANNUAL
q4.sort_values('exceedance_rate_24h_%', ascending=False)

,n_readings,exceed_24h,exceedance_rate_24h_%,annual_mean_pm25,exceeds_annual_guideline
city,,,,,
johannesburg,3,3,100.00,44.70,True
kigali,1418,1377,97.11,38.93,True
kampala,7113,6798,95.57,52.58,True
addis_ababa,2539,2217,87.32,26.10,True
lagos,9042,7096,78.48,38.62,True
nairobi,2108,1284,60.91,23.15,True


In [33]:
worst_reliable = q4.drop(index='johannesburg')['exceedance_rate_24h_%'].idxmax()
print(f"Answer: {worst_reliable.replace('_',' ').title()} exceeds the WHO 24-hour guideline (15 µg/m³) most often "
      f"among cities with adequate sample size "
      f"({q4.loc[worst_reliable, 'exceedance_rate_24h_%']:.1f}% of {q4.loc[worst_reliable, 'n_readings']:,} readings).")
print(f"Note: Johannesburg shows {q4.loc['johannesburg', 'exceedance_rate_24h_%']:.0f}% exceedance but from only "
      f"{q4.loc['johannesburg', 'n_readings']:.0f} readings — not statistically reliable, flagged for monitoring expansion instead.")
n_all_exceed = q4['exceeds_annual_guideline'].sum()
print(f"All {n_all_exceed} of {len(q4)} cities have an annual mean PM2.5 above the WHO annual guideline of 5 µg/m³.")

Answer: Kigali exceeds the WHO 24-hour guideline (15 µg/m³) most often among cities with adequate sample size (97.1% of 1,418 readings).
Note: Johannesburg shows 100% exceedance but from only 3 readings — not statistically reliable, flagged for monitoring expansion instead.
All 6 of 6 cities have an annual mean PM2.5 above the WHO annual guideline of 5 µg/m³.


## 5. Which monitoring stations have the most complete and reliable datasets?
**Technique: data-completeness scoring (composite of coverage ratio, quality band, and flag rate)**

In [34]:
quality_map = {'high': 3, 'medium': 2, 'low': 1, 'very low': 0}

station = df.groupby(['city', 'station_id', 'station_name']).agg(
    n_records=('pm25', 'count'),
    avg_coverage_ratio=('coverage_ratio', 'mean'),
    pct_interpolated=('value_interpolated_flag', 'mean'),
    pct_flagged=('has_flags', 'mean'),
    quality_band_mode=('quality_band', lambda s: s.mode().iat[0]),
).reset_index()

station['quality_score'] = station['quality_band_mode'].map(quality_map)
# Composite reliability score: high coverage, low interpolation/flagging, high quality band, more records
station['reliability_score'] = (
    station['avg_coverage_ratio'] * 0.4
    + (station['quality_score'] / 3) * 0.3
    + (1 - station['pct_interpolated']) * 0.15
    + (1 - station['pct_flagged']) * 0.15
)
q5 = station.sort_values('reliability_score', ascending=False)
q5[['city', 'station_name', 'n_records', 'avg_coverage_ratio', 'quality_band_mode', 'reliability_score']].head(10)

,city,station_name,n_records,avg_coverage_ratio,quality_band_mode,reliability_score
2,addis_ababa,koshe test,1,1.00,high,1.00
4,addis_ababa,The Urban Center Office,1,1.00,high,1.00
3,addis_ababa,office test,1,1.00,high,1.00
5,addis_ababa,koshe outdoor 1,1,1.00,high,1.00
6,addis_ababa,koshe out 2,2,1.00,high,1.00
7,johannesburg,Little John Park,3,1.00,high,1.00
44,kigali,Kigali_US_Embassy,267,1.00,high,1.00
45,kigali,Kigali_Rwanda,267,1.00,high,1.00
42,kampala,AirQo testing LAB,9,1.00,high,1.00
102,lagos,Iganmu Blueline Train Station,24,1.00,high,1.00


In [35]:
best = q5.iloc[0]
print(f"Answer: {best['station_name']} ({best['city'].replace('_',' ').title()}) is the most reliable "
      f"station (reliability score={best['reliability_score']:.2f}, coverage={best['avg_coverage_ratio']:.1%}, "
      f"quality band='{best['quality_band_mode']}').")
print("\nAvg reliability score by city:")
print(q5.groupby('city')['reliability_score'].mean().sort_values(ascending=False))

Answer: koshe test (Addis Ababa) is the most reliable station (reliability score=1.00, coverage=100.0%, quality band='high').

Avg reliability score by city:
city
johannesburg   1.00
addis_ababa    0.95
kigali         0.89
kampala        0.86
nairobi        0.85
lagos          0.79
Name: reliability_score, dtype: float64


## 6. Which cities should be prioritised for air-quality intervention, and why?
**Technique: synthesis via correlation + composite ranking across Q1, Q3, Q4, Q5 outputs**

In [36]:
synth = pd.DataFrame({
    "avg_pm25": q1["mean_pm25"],
    "seasonal_range": seasonal_range,
    "exceedance_rate_24h_%": q4["exceedance_rate_24h_%"],
    "avg_station_reliability": q5.groupby("city")["reliability_score"].mean(),
    "n_stations": q2["n_stations"],
    "n_readings": q4["n_readings"],
})

# Minimum evidence rule for intervention ranking.
synth["reliable_for_priority"] = synth["n_readings"] >= 30

# Spearman correlation among reliable cities only.
reliable_subset = synth[synth["reliable_for_priority"]]
corr, p = stats.spearmanr(reliable_subset["avg_pm25"], reliable_subset["avg_station_reliability"] )
print(f"Spearman correlation (avg PM2.5 vs avg station reliability, reliable cities): rho={corr:.2f}, p={p:.3f}")

# Priority score: pollution severity and exceedance weighted highest.
for col in ["avg_pm25", "seasonal_range", "exceedance_rate_24h_%"]:
    synth[col + "_z"] = (synth[col] - synth[col].mean()) / synth[col].std()

synth["priority_score"] = (
    synth["avg_pm25_z"] * 0.5 + synth["exceedance_rate_24h_%_z"] * 0.35 + synth["seasonal_range_z"] * 0.15
)

q6 = synth[synth["reliable_for_priority"]].sort_values("priority_score", ascending=False)
q6_watchlist = synth[~synth["reliable_for_priority"]].sort_values("n_readings")

q6[["avg_pm25", "exceedance_rate_24h_%", "seasonal_range", "avg_station_reliability", "n_readings", "priority_score"]]

Spearman correlation (avg PM2.5 vs avg station reliability, reliable cities): rho=0.10, p=0.873


,avg_pm25,exceedance_rate_24h_%,seasonal_range,avg_station_reliability,n_readings,priority_score
city,,,,,,
kampala,52.58,95.57,27.68,0.86,7113,0.93
kigali,38.93,97.11,30.82,0.89,1418,0.39
lagos,38.62,78.48,41.37,0.79,9042,0.04
addis_ababa,26.10,87.32,18.86,0.95,2539,-0.55
nairobi,23.15,60.91,29.55,0.85,2108,-1.19


In [37]:
print("Answer: Priority order for intervention (highest first, reliable evidence only):")
for i, (city, row) in enumerate(q6.iterrows(), 1):
    print(f"{i}. {city.replace('_', ' ').title()} — avg PM2.5={row['avg_pm25']:.1f} µg/m³, "
          f"exceeds WHO 24h guideline {row['exceedance_rate_24h_%']:.0f}% of the time, "
          f"seasonal range={row['seasonal_range']:.1f} µg/m³, n={int(row['n_readings'])}")

if len(q6_watchlist) > 0:
    print("\nMonitoring-expansion watchlist (excluded from priority ranking due to low sample size):")
    for city, row in q6_watchlist.iterrows():
        print(f"- {city.replace('_',' ').title()}: n={int(row['n_readings'])} readings, "
              f"{int(row['n_stations'])} station(s)")

Answer: Priority order for intervention (highest first, reliable evidence only):
1. Kampala — avg PM2.5=52.6 µg/m³, exceeds WHO 24h guideline 96% of the time, seasonal range=27.7 µg/m³, n=7113
2. Kigali — avg PM2.5=38.9 µg/m³, exceeds WHO 24h guideline 97% of the time, seasonal range=30.8 µg/m³, n=1418
3. Lagos — avg PM2.5=38.6 µg/m³, exceeds WHO 24h guideline 78% of the time, seasonal range=41.4 µg/m³, n=9042
4. Addis Ababa — avg PM2.5=26.1 µg/m³, exceeds WHO 24h guideline 87% of the time, seasonal range=18.9 µg/m³, n=2539
5. Nairobi — avg PM2.5=23.2 µg/m³, exceeds WHO 24h guideline 61% of the time, seasonal range=29.6 µg/m³, n=2108

Monitoring-expansion watchlist (excluded from priority ranking due to low sample size):
- Johannesburg: n=3 readings, 1 station(s)


## Summary of statistical techniques applied
| Technique | Question(s) | Purpose |
|---|---|---|
| Descriptive statistics & ranking | Q1, Q2 | Rank cities by mean/median PM2.5 and monitoring intensity |
| Time-series/seasonal analysis + one-way ANOVA | Q3 | Quantify and test significance of within-city monthly variation |
| Threshold/exceedance-rate analysis | Q4 | % of readings above WHO 24-hr and annual guidelines |
| Composite completeness scoring | Q5 | Rank stations by coverage, quality band, flag/interpolation rate |
| Spearman correlation + weighted composite ranking | Q6 | Synthesize Q1–Q5 evidence into an intervention-priority order |


## 6. Visualizations

**Owner:** Gloria

**TODO:** At least 5 charts, each with a one-line justification for the chart type chosen. Suggested: city PM2.5 ranking, pollutant frequency by city, seasonal variation, WHO exceedance over time, station completeness, station location map.

In [158]:
# TODO (Gloria): 6. Visualizations


## 7. Findings & Recommendations

**Owner:** Gloria (with input from all)

**TODO:** Evidence-backed answers to all 6 client questions, each citing the specific stat/chart that supports it. Prioritization recommendation for UNEP: which cities need intervention, and why.

In [159]:
# TODO (Gloria (with input from all)): 7. Findings & Recommendations
